In [1]:
import pandas as pd
import numpy as np
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.cluster import KMeans
import matplotlib.pyplot as plt

In [3]:
df = pd.read_csv('twcs.csv')
df.head()

,tweet_id,author_id,inbound,created_at,text,response_tweet_id,in_response_to_tweet_id
0,1,sprintcare,False,Tue Oct 31 22:10:47 +0000 2017,@115712 I understand. I would like to assist y...,2,3.0
1,2,115712,True,Tue Oct 31 22:11:45 +0000 2017,@sprintcare and how do you propose we do that,NaN,1.0
2,3,115712,True,Tue Oct 31 22:08:27 +0000 2017,@sprintcare I have sent several private messag...,1,4.0
3,4,sprintcare,False,Tue Oct 31 21:54:49 +0000 2017,@115712 Please send us a Private Message so th...,3,5.0
4,5,115712,True,Tue Oct 31 21:49:35 +0000 2017,@sprintcare I did.,4,6.0


In [4]:
df = df[df["inbound"] == True]
print(df.shape)
print(df["text"].head(10))

(1537843, 7)
1         @sprintcare and how do you propose we do that
2     @sprintcare I have sent several private messag...
4                                    @sprintcare I did.
6             @sprintcare is the worst customer service
8     @sprintcare You gonna magically change your co...
10    @sprintcare Since I signed up with you....Sinc...
12    @115714 y’all lie about your “great” connectio...
14    @115714 whenever I contact customer support, t...
16    @Ask_Spectrum Would you like me to email you a...
18    @Ask_Spectrum I received this from your corpor...
Name: text, dtype: object


In [5]:
import re

def clean_text(text):
    text = text.lower()
    text = re.sub(r'@\w+', '', text)
    text = re.sub(r'[^\w\s]', '', text)
    text = text.strip()
    return text

df["cleaned_text"] = df["text"].apply(clean_text)
print(df[["text", "cleaned_text"]].head(5))

                                                text  \
1      @sprintcare and how do you propose we do that   
2  @sprintcare I have sent several private messag...   
4                                 @sprintcare I did.   
6          @sprintcare is the worst customer service   
8  @sprintcare You gonna magically change your co...   

                                        cleaned_text  
1                  and how do you propose we do that  
2  i have sent several private messages and no on...  
4                                              i did  
6                      is the worst customer service  
8  you gonna magically change your connectivity f...  


In [6]:
keywords = ['order', 'food', 'items', 'dasher', 'refund', 'doordash']

mask = df['cleaned_text'].str.contains('|'.join(keywords), case=False, na=False)

df_food = df[mask]

print(df_food.shape)
print(df_food['cleaned_text'].head(5))

(97495, 8)
78     so frustrated with   ordered dinner on saturda...
88     this is nasty a used napkin in someones food h...
116    arent require charge 4 a bag\npaper bags\nshop...
119    excellent halloweening   countcolin halloween ...
145                   i pre ordered my game at game stop
Name: cleaned_text, dtype: object


In [7]:
print(df_food['cleaned_text'].isnull().sum())

0


In [8]:
from sklearn.feature_extraction.text import TfidfVectorizer

vectorizer = TfidfVectorizer(max_features=1000, stop_words='english')

X = vectorizer.fit_transform(df_food['cleaned_text'])

print(X.shape)

(97495, 1000)


In [9]:
from sklearn.cluster import KMeans

kmeans = KMeans(n_clusters=6, random_state=42, n_init=10)

kmeans.fit(X)

df_food['cluster'] = kmeans.labels_

print(df_food['cluster'].value_counts())

c:\Softwares\anaconda\Lib\site-packages\joblib\externals\loky\backend\context.py:136: UserWarning: Could not find the number of physical cores for the following reason:
[WinError 2] The system cannot find the file specified
Returning the number of logical cores instead. You can silence this warning by setting LOKY_MAX_CPU_COUNT to the number of cores you want to use.
  warnings.warn(
  File "c:\Softwares\anaconda\Lib\site-packages\joblib\externals\loky\backend\context.py", line 257, in _count_physical_cores
    cpu_info = subprocess.run(
               ^^^^^^^^^^^^^^^
  File "c:\Softwares\anaconda\Lib\subprocess.py", line 548, in run
    with Popen(*popenargs, **kwargs) as process:
         ^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "c:\Softwares\anaconda\Lib\subprocess.py", line 1026, in __init__
    self._execute_child(args, executable, preexec_fn, close_fds,
  File "c:\Softwares\anaconda\Lib\subprocess.py", line 1538, in _execute_child
    hp, ht, pid, tid = _winapi.CreateProcess(executable

cluster
5    31085
2    20895
0    17445
1    12645
3     9386
4     6039
Name: count, dtype: int64


C:\Users\rouna\AppData\Local\Temp\ipykernel_3032\13932040.py:7: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_food['cluster'] = kmeans.labels_


In [10]:
df_food = df_food.copy()
df_food['cluster'] = kmeans.labels_

In [11]:
def get_top_words(cluster_num, n_words=10):
    cluster_docs = df_food[df_food['cluster'] == cluster_num]['cleaned_text']
    vectorizer2 = TfidfVectorizer(max_features=1000, stop_words='english')
    X2 = vectorizer2.fit_transform(cluster_docs)
    scores = X2.mean(axis=0).A1
    words = vectorizer2.get_feature_names_out()
    top_indices = scores.argsort()[-n_words:][::-1]
    return [words[i] for i in top_indices]

for i in range(6):
    print(f"Cluster {i}: {get_top_words(i)}")

Cluster 0: ['refund', 'want', 'money', 'just', 'need', 'order', 'got', 'dont', 'help', 'im']
Cluster 1: ['ordered', 'delivery', 'got', 'order', 'just', 'delivered', 'today', 'amazon', 'day', 'prime']
Cluster 2: ['items', 'orders', 'preorder', 'refunded', 'ordering', 'just', 'preordered', 'delivery', 'im', 'time']
Cluster 3: ['food', 'just', 'order', 'im', 'got', 'time', 'amp', 'dont', 'ordered', 'driver']
Cluster 4: ['service', 'customer', 'order', 'refund', 'amazon', 'delivery', 'worst', 'care', 'delivered', 'just']
Cluster 5: ['order', 'delivery', 'delivered', 'just', 'help', 'amazon', 'time', 'number', 'im', 'placed']


In [16]:
!pip install groq


[notice] A new release of pip is available: 25.1.1 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [18]:
from groq import Groq

client = Groq(api_key="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")

In [23]:
def generate_postmortem(cluster_num, n_samples=50):
    samples = df_food[df_food['cluster'] == cluster_num]['cleaned_text'].sample(n_samples, random_state=42).tolist()
    complaints_text = "\n".join(samples)
    
    chat_completion = client.chat.completions.create(
        model="llama-3.3-70b-versatile",
        max_tokens=300,
        messages=[{
            "role": "user",
            "content": f"""You are a customer experience analyst at a food delivery company.
            
Here are {n_samples} real customer complaints from the same category:

{complaints_text}

In 3 sentences:
1. What is the root cause?
2. What operational fix would you recommend?"""
        }]
    )
    return chat_completion.choices[0].message.content

In [24]:
postmortem = generate_postmortem(0)
print("CLUSTER 0 - Refund Requests")
print("="*50)
print(postmortem)

CLUSTER 0 - Refund Requests
The root cause of the customer complaints is the inefficiency and lack of transparency in the refund process, with many customers experiencing delays, difficulties, or denials in obtaining refunds for various issues such as cancelled orders, faulty products, or unsatisfactory services. To address this, I would recommend an operational fix that streamlines and automates the refund process, including clear and timely communication with customers, a user-friendly refund request system, and a dedicated team to handle refund-related issues. Additionally, implementing a refund policy that is fair, flexible, and easy to understand, and providing regular updates on the status of refunds, can help to reduce customer frustration and improve overall satisfaction with the company's refund process.


In [25]:
cluster_names = {
    0: "Refund Requests",
    1: "Late or Missing Delivery", 
    2: "Pre-order Issues",
    3: "Food Delivery Complaints",
    4: "Poor Customer Service",
    5: "Delivery Status Issues"
}

postmortems = {}

for i in range(6):
    print(f"Generating postmortem for Cluster {i}...")
    postmortems[i] = generate_postmortem(i)
    print(f"✓ Done\n")

print("All postmortems generated!")

Generating postmortem for Cluster 0...
✓ Done

Generating postmortem for Cluster 1...
✓ Done

Generating postmortem for Cluster 2...
✓ Done

Generating postmortem for Cluster 3...
✓ Done

Generating postmortem for Cluster 4...
✓ Done

Generating postmortem for Cluster 5...
✓ Done

All postmortems generated!


In [26]:
for i in range(6):
    print(f"\nCLUSTER {i} - {cluster_names[i]}")
    print("="*50)
    print(postmortems[i])
    print()


CLUSTER 0 - Refund Requests
The root cause of these customer complaints appears to be a lack of transparency and efficiency in the refund process, with many customers experiencing delays, difficulties, or denials of refunds for various reasons, including cancelled orders, defective products, and disputed transactions. To address this issue, I would recommend implementing a streamlined and automated refund process that provides clear guidelines and timelines for customers to initiate and track their refunds, as well as ensuring that customer support teams are empowered to resolve refund-related issues quickly and fairly. Additionally, companies should consider providing regular updates and notifications to customers on the status of their refunds, and offering alternative solutions or compensation when refunds are not possible, to improve customer satisfaction and reduce frustration.


CLUSTER 1 - Late or Missing Delivery
The root cause of the customer complaints appears to be a combin